# HumMobCov — Main Analysis Notebook

Structured in three sections:
1. **INPUT** — choose the region, load the dataset, inspect configuration
2. **MAIN** — run the full processing pipeline (per-user metric computation)
3. **VISUALIZATION** — produce all plots from the saved per-user results

All paths, parameters and flags are controlled through:
- `src/constants.py` — global constants and directory layout
- `data/config/config_<REGION>.json` — feature flags (which metrics to compute)

---
## 0 · Imports

In [1]:
import sys
from pathlib import Path

# Make the src package importable regardless of working directory
PROJECT_ROOT = Path().resolve().parent   # HumMobCov/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Central import hub — brings in numpy, pandas, skmob, matplotlib, etc.
# and all project classes / constants
from src import (
    # constants
    PROJECT_ROOT, DIR_SRC, DIR_OUTPUT, DIR_DATA, DIR_CONFIG,
    PERIOD_NAMES, PERIOD_DIVISION, PERIOD_NAMES_TO_DIVISION,
    MIN_POINTS_PER_USER, TIME_THRESHOLD_HOURS, US_BOUNDING_BOX,
    RURALITY_LEVELS, PARTY_NAMES, K_RADIUS_VALUES, LIST_REGIONS,
    # dataset classes
    DataSet_California, DataSet_Massachusets, dataset_info,
    # pipeline
    analyze_from_dataset, analyze_from_s3_progressive, compute_all, get_config,
    # user
    User,
    # plotter
    plotter,
    # storage
    ParquetStore,
    # utilities
    get_already_saved_user_per_period, ifnotexistsmkdir,
)
from src.constants import (
    DIR_MILESTONES_SERVER,
    S3_ENDPOINT_URL, S3_BUCKET, S3_RAW_PREFIX, DIR_SHARD_TEMP,
    S3_OUTPUT_BUCKET, S3_OUTPUT_PREFIX,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Project root:', PROJECT_ROOT)
print('Output dir:  ', DIR_OUTPUT)
print('Data dir:    ', DIR_DATA)
print('S3 endpoint: ', S3_ENDPOINT_URL)
print('S3 bucket:   ', S3_BUCKET)
print('Out bucket:  ', S3_OUTPUT_BUCKET)


Project root: /home/aamaduzzi/HumMobCov
Output dir:   /home/aamaduzzi/HumMobCov/output
Data dir:     /home/aamaduzzi/HumMobCov/data
S3 endpoint:  https://s3.atlas.fbk.eu
S3 bucket:    chub-datalake
Out bucket:   chub-datalake


---
## 1 · INPUT

**Edit this section** to choose which region to analyse and to review / override any parameter.

In [2]:
# ─── Choose region ──────────────────────────────────────────────────────────
# Options: "CA"  (California)  or  "MA"  (Massachusetts)
REGION = "CA"

assert REGION in LIST_REGIONS, f"Unknown region '{REGION}'. Choose from {LIST_REGIONS}"

In [3]:
# ─── Optional parameter overrides ───────────────────────────────────────────
# Leave as None to use the defaults from constants.py

OVERRIDE_NP_            = None   # e.g. 30  (minimum stops per user per period)
OVERRIDE_T_THRESHOLD    = None   # e.g. 2   (minimum hours between stops)
OVERRIDE_OUTPUT_DIR     = None   # e.g. Path('/my/custom/output')
OVERRIDE_CONFIG_DIR     = None   # e.g. Path('/my/configs')

In [4]:
# ─── Initialise dataset ─────────────────────────────────────────────────────
if REGION == "CA":
    dataset = DataSet_California()
elif REGION == "MA":
    dataset = DataSet_Massachusets()

# Apply overrides
if OVERRIDE_NP_ is not None:
    dataset.np_ = OVERRIDE_NP_
if OVERRIDE_T_THRESHOLD is not None:
    dataset.t_threshold = OVERRIDE_T_THRESHOLD

print(f"Region:              {dataset.id_}")
print(f"Min points (np_):    {dataset.np_}")
print(f"Time threshold (h):  {dataset.t_threshold}")
print(f"Output directory:    {dataset.dir_output}")
print(f"Number of raw files: {len(dataset.dir_files)}")

Region:              CA
Min points (np_):    20
Time threshold (h):  1
Output directory:    /home/aamaduzzi/HumMobCov/milestones_analysis/CA/dataxuser
Number of raw files: 0


In [5]:
# ─── Preview algorithm-flow configuration ───────────────────────────────────
import json

cfg = get_config(REGION, config_dir=OVERRIDE_CONFIG_DIR)
print("Active feature flags:")
for key, val in cfg.items():
    if not key.startswith('_'):
        flag = '✓' if val else '✗'
        print(f"  {flag}  {key}")

Active feature flags:
  ✓  raw_trajectories
  ✓  is_radius_gyration
  ✗  already_computed_rg
  ✓  is_weekly_radius_gyration
  ✓  is_gonzalez
  ✗  already_computed_gonzalez
  ✓  is_random_entropy
  ✗  already_computed_random_entropy
  ✓  is_uncorrelated_entropy
  ✗  already_computed_uncorrelated_entropy
  ✓  is_real_entropy
  ✗  already_computed_real_entropy
  ✓  is_distance
  ✗  already_computed_distance
  ✓  is_home
  ✗  already_computed_home
  ✓  is_krg
  ✗  already_computed_krg
  ✓  is_St
  ✗  already_computed_St
  ✓  is_fraction_time
  ✗  already_computed_fraction_time
  ✓  is_county_rural
  ✗  already_computed_county_rural
  ✓  is_frequency
  ✗  already_computed_frequency
  ✗  is_week2points
  ✓  save_results


In [6]:
# ─── Preview time periods ────────────────────────────────────────────────────
print("Analysis periods:")
for name, (start, end) in PERIOD_NAMES_TO_DIVISION.items():
    print(f"  {name:25s}  {start.date()}  →  {end.date()}")

Analysis periods:
  15 jan - 15 march          2020-01-15  →  2020-03-15
  15 march - 15 may          2020-03-15  →  2020-05-15
  15 may - sept              2020-05-15  →  2020-09-30


---
## 2 · MAIN

Runs the full per-user computation pipeline, selecting the appropriate execution mode automatically:

| Mode | Condition | Action |
|---|---|---|
| **A — local raw** | Raw parquet shards present on local filesystem | `analyze_from_dataset()` — standard batch processing |
| **B — S3 progressive** | `raw_trajectories=true` in config but files not local | `analyze_from_s3_progressive()` — download one shard at a time, compute, delete |
| **C-CA — legacy migration** | CA only; per-user CSV.gz files in `dataxuser/` | `store.migrate_all_periods()` — one-time migration to parquet |
| **C-MA — legacy migration** | MA only; per-metric shard dirs in `milestones_analysis/MA/` | `store.migrate_all_periods_MA()` — one-time migration to parquet |
| **Done** | All periods already in parquet store | Skip — go to Section 3 |

**The pipeline is resume-safe at two levels:**
- *Shard level* (S3 mode): `shard_checkpoint_np_{np_}_t_{t}.json` records which raw files have been fully processed. Interrupted runs restart from the next unprocessed shard.
- *User level*: users already present in the parquet store are detected in O(1) via footer metadata and skipped unconditionally.

**S3 configuration** (override via environment variables if needed):
```
S3_ENDPOINT_URL  https://s3.atlas.fbk.eu   (env: S3_ENDPOINT_URL)
S3_BUCKET        chub-datalake              (env: S3_BUCKET)
S3_PREFIX_CA     shared/cuebiq/MOBS/urban_rural_flow_stops_cali_urban_rural_v3  (env: S3_PREFIX_CA)
S3_PREFIX_MA     shared/cuebiq/MOBS/20220330_stops_hq_users_MA                  (env: S3_PREFIX_MA)
SHARD_TEMP_DIR   HumMobCov/.shard_tmp       (env: SHARD_TEMP_DIR)
```


In [ ]:

# ─── Compute-path settings ───────────────────────────────────────────────────
# USE_VECTORIZED=True  → polars/numpy batch path (10-100x faster for most metrics).
#   - Scalar metrics (RG, entropy, distance, home, k-RG, frequency): fully
#     parallel via polars' internal thread pool (uses ALL CPU cores automatically).
#   - Python-callback metrics (real_entropy, S(t), Gonzalez): run serially;
#     parallelization was benchmarked and shown to be slower due to IPC overhead.
# USE_VECTORIZED=False → legacy skmob per-user loop (slower, kept for reference).
USE_VECTORIZED = True

# ─── Initialise parquet store ────────────────────────────────────────────────
store = ParquetStore(
    base_dir     = DIR_MILESTONES_SERVER / REGION,
    np_          = dataset.np_,
    t_threshold  = dataset.t_threshold,
)

# ─── Check what is already computed ──────────────────────────────────────────
# IMPORTANT: completion semantics depend on the execution mode.
#
# raw_trajectories=true (S3 / local raw modes):
#   The unit of work is a raw SHARD.  Completion = every shard in the S3
#   prefix appears in shard_checkpoint_np_{np_}_t_{t}.json.
#   We CANNOT infer this from store contents alone — a partial run (1-of-N
#   shards) produces a fully-consistent store that would look "done".
#   → Always enter the pipeline; analyze_from_s3_progressive /
#     analyze_from_dataset manage their own shard-level resume internally.
#
# raw_trajectories=false (legacy migration):
#   The unit of work is a USER.  Resume is user-granular: already-migrated
#   users are skipped inside migrate_all_periods / migrate_all_periods_MA.
#   → A period is "done" only if all_scalars has any users (safe proxy,
#     because migration never runs unless a period folder is missing).

use_shard_resume = bool(cfg.get("raw_trajectories"))

if use_shard_resume:
    # Always enter the pipeline; inner shard-level resume handles skipping.
    periods_done = []
    periods_todo = list(PERIOD_NAMES)
else:
    periods_done = [
        p for p in PERIOD_NAMES
        if len(store.get_computed_users(p, "all_scalars")) > 0
    ]
    periods_todo = [p for p in PERIOD_NAMES if p not in periods_done]

print(f"Periods already in store  : {periods_done or 'none'}")
print(f"Periods still to compute  : {periods_todo or 'none'}")
print(f"Compute path              : {'vectorized (polars)' if USE_VECTORIZED else 'legacy (skmob)'}")
print(f"Output bucket             : {S3_OUTPUT_BUCKET}")
print(f"Output prefix             : {S3_OUTPUT_PREFIX[REGION]}")

if not periods_todo:
    print("\nParquet store already populated for all periods. Nothing to do.")

else:
    # ── CASE A: local raw parquet files ──────────────────────────────────────
    local_raw_files = [f for f in dataset.dir_files if Path(f).exists()]

    if cfg.get("raw_trajectories") and local_raw_files:
        print(f"\nMODE A — local raw data ({len(local_raw_files)} shards found).")
        analyze_from_dataset(
            dataset,
            region         = REGION,
            config_dir     = OVERRIDE_CONFIG_DIR,
            output_dir     = OVERRIDE_OUTPUT_DIR,
            store          = store,
            batch_size     = 20000,
            use_vectorized = USE_VECTORIZED,
        )
        print("Computation complete. Consolidating shards…")
        for p in dataset.period_names:
            store.consolidate_all(p)
        print("Uploading results to S3…")
        store.upload_all_to_s3(
            period_names      = dataset.period_names,
            s3_bucket         = S3_OUTPUT_BUCKET,
            s3_prefix         = S3_OUTPUT_PREFIX[REGION],
            endpoint_url      = S3_ENDPOINT_URL,
            delete_after      = False,
            consolidate_first = False,
        )
        print("Done.")

    # ── CASE B: S3 progressive (download one shard → compute → upload → delete) ──
    elif cfg.get("raw_trajectories"):
        print(f"\nMODE B — S3 progressive download.")
        print(f"  Input endpoint  : {S3_ENDPOINT_URL}")
        print(f"  Input bucket    : {S3_BUCKET}")
        print(f"  Input prefix    : {S3_RAW_PREFIX[REGION]}")
        print(f"  Temp dir        : {DIR_SHARD_TEMP}")
        print(f"  Output bucket   : {S3_OUTPUT_BUCKET}")
        print(f"  Output prefix   : {S3_OUTPUT_PREFIX[REGION]}")
        print(f"  Periods         : {periods_todo}\n")
        analyze_from_s3_progressive(
            dataset,
            region                    = REGION,
            cfg                       = cfg,
            store                     = store,
            endpoint_url              = S3_ENDPOINT_URL,
            bucket                    = S3_BUCKET,
            s3_prefix                 = S3_RAW_PREFIX[REGION],
            temp_dir                  = DIR_SHARD_TEMP,
            batch_size                = 20000,
            use_vectorized            = USE_VECTORIZED,
            output_endpoint_url       = S3_ENDPOINT_URL,
            output_bucket             = S3_OUTPUT_BUCKET,
            output_s3_prefix          = S3_OUTPUT_PREFIX[REGION],
            delete_local_after_upload = True,
        )

    # ── CASE C-CA: migrate legacy dataxuser per-user CSV.gz files ────────────
    elif REGION == "CA":
        legacy_dir = DIR_MILESTONES_SERVER / REGION / "dataxuser"
        if legacy_dir.exists() and any(legacy_dir.iterdir()):
            print(
                f"\nMODE C-CA — migrating legacy dataxuser/ CSV.gz files.\n"
                f"  Periods missing: {periods_todo}\n"
                f"  (Already-migrated users are skipped — safe to re-run.)"
            )
            store.migrate_all_periods(
                legacy_dir   = legacy_dir,
                period_names = periods_todo,
                np_          = dataset.np_,
                t            = dataset.t_threshold,
                batch_size   = 20000,
                consolidate  = True,
            )
            print("Uploading results to S3…")
            store.upload_all_to_s3(
                period_names      = dataset.period_names,
                s3_bucket         = S3_OUTPUT_BUCKET,
                s3_prefix         = S3_OUTPUT_PREFIX[REGION],
                endpoint_url      = S3_ENDPOINT_URL,
                delete_after      = False,
                consolidate_first = False,
            )
        else:
            print(
                f"\nCA: no dataxuser/ directory found at {legacy_dir}.\n"
                f"Set raw_trajectories=true in config_CA.json to compute from S3."
            )

    # ── CASE C-MA: migrate legacy per-metric shard directories ───────────────
    elif REGION == "MA":
        ma_legacy_base = DIR_MILESTONES_SERVER / "MA"
        print(
            f"\nMODE C-MA — migrating legacy metric-folder shard files.\n"
            f"  Periods missing: {periods_todo}\n"
            f"  np_={dataset.np_}, t={dataset.t_threshold}\n"
            f"  (Already-migrated users are skipped — safe to re-run.)"
        )
        store.migrate_all_periods_MA(
            ma_legacy_base = ma_legacy_base,
            period_names   = periods_todo,
            np_            = dataset.np_,
            t              = dataset.t_threshold,
            batch_size     = 20000,
            consolidate    = True,
        )
        print("Uploading results to S3…")
        store.upload_all_to_s3(
            period_names      = dataset.period_names,
            s3_bucket         = S3_OUTPUT_BUCKET,
            s3_prefix         = S3_OUTPUT_PREFIX[REGION],
            endpoint_url      = S3_ENDPOINT_URL,
            delete_after      = False,
            consolidate_first = False,
        )

    else:
        print(f"Unknown region {REGION!r}. No action taken.")


Logical CPUs (os.cpu_count) : 96
Cgroup CPU quota            : 4  ← used as N_WORKERS
Periods already in store  : none
Periods still to compute  : ['15 jan - 15 march', '15 march - 15 may', '15 may - sept']
Compute path              : vectorized (polars)
Worker processes          : 4  [logical: 96, cgroup quota: 4]
Output bucket             : chub-datalake
Output prefix             : final_pipeline/CA

MODE B — S3 progressive download.
  Input endpoint  : https://s3.atlas.fbk.eu
  Input bucket    : chub-datalake
  Input prefix    : shared/cuebiq/MOBS/urban_rural_flow_stops_cali_urban_rural_v3
  Temp dir        : /home/aamaduzzi/HumMobCov/.shard_tmp
  Output bucket   : chub-datalake
  Output prefix   : final_pipeline/CA
  Periods         : ['15 jan - 15 march', '15 march - 15 may', '15 may - sept']



TypeError: analyze_from_s3_progressive() got an unexpected keyword argument 'n_workers'

In [ ]:
# ─── Pipeline summary ────────────────────────────────────────────────────────
# Show how many users are available per period and metric kind.
# Reads only parquet footer metadata (no data loaded).

print("Users available in parquet store:")
for period in PERIOD_NAMES:
    scalars = store.get_computed_users(period, "all_scalars")
    gonzalez = store.get_computed_users_long(period, "gonzalez")
    st = store.get_computed_users(period, "S")
    wrg = store.get_computed_users(period, "weekly_rg")
    freq = store.get_computed_users_long(period, "frequency")
    print(
        f"  {period:25s}  "
        f"scalars={len(scalars):>7,}  "
        f"gonzalez={len(gonzalez):>7,}  "
        f"S(t)={len(st):>7,}  "
        f"weekly_rg={len(wrg):>7,}  "
        f"freq={len(freq):>7,}"
    )


Users available in parquet store:
  15 jan - 15 march          scalars=1,438,004  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0
  15 march - 15 may          scalars=      0  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0
  15 may - sept              scalars=      0  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0


---
## 3 · VISUALIZATION

All plots are produced from the already-saved per-user files.
Run this section independently of Section 2 once results are on disk.

In [ ]:
# ─── Initialise plotter ──────────────────────────────────────────────────────
# Pass `store=store` to use the fast parquet backend.
# Omit it (or pass None) to fall back to the legacy per-file mode.

plt_obj = plotter(
    np_            = dataset.np_,
    period_division= dataset.period_division,
    period_names   = dataset.period_names,
    t_threshold    = dataset.t_threshold,
    region         = REGION,
    county2party   = dataset.county2party,
    df_rurality    = dataset.df_rurality,
    output_dir     = OVERRIDE_OUTPUT_DIR,
    store          = store,          # ← parquet-store mode
)

# Show user counts per period (from parquet store metadata)
print("Users available per period (from parquet store):")
for period in plt_obj.period_names:
    n = len(store.get_computed_users(period, "all_scalars"))
    print(f"  {period:25s}  {n:>8,} users")


Users available per period (from parquet store):
  15 jan - 15 march          1,438,004 users
  15 march - 15 may                 0 users
  15 may - sept                     0 users


### 3.1 Radius of Gyration

In [ ]:
plt_obj.plot_rg()

In [ ]:
plt_obj.plot_rg_party_per_period()

In [ ]:
plt_obj.plot_rg_rurality_per_period()

### 3.2 Weekly Radius of Gyration

In [ ]:
plt_obj.plot_weekly_rg()

In [ ]:
plt_obj.plot_rg_rurality_weekly()

In [ ]:
plt_obj.plot_rg_party_weekly()

### 3.3 k-Radius of Gyration

In [ ]:
plt_obj.plot_krg()

### 3.4 Distance

In [ ]:
plt_obj.plot_distance()

### 3.5 Entropy

In [ ]:
plt_obj.plot_entropy()

### 3.6 Exploration Curve S(t)

In [ ]:
plt_obj.plot_St()

### 3.7 Location Frequency

In [ ]:
plt_obj.plot_frequency()

### 3.8 Gonzalez Trajectory Shape

In [ ]:
plt_obj.plot_gonzalez()

In [ ]:
plt_obj.plot_sigmaxy()

---
## 4 · GAP ANALYSIS — Methodological improvements

Four standalone plot functions that address scientific gaps in the paper:

| # | Gap | Function |
|---|-----|----------|
| 1 | **Causal framing** — NPIs vs. voluntary behaviour | `plot_gap1_npi_timeline` |
| 2 | **Sampling bias** — opt-in skew quantified | `plot_gap2_sampling_bias` |
| 3 | **Party/rurality conflation** — partial correlations + OLS | `plot_gap3_party_rurality` |
| 4 | **Post-lockdown asymmetry** — explicitly shown in Results | `plot_gap4_post_lockdown_asymmetry` |

Each bridge method on `plt_obj` delegates to a standalone function in
`src/gap_analysis_plots.py` — you can also call those functions directly
if you have a custom DataFrame.


### 4.1 Gap 1 — Causal framing: NPI event timeline

Overlays formal government-order dates on the weekly RG time-series.
If mobility started dropping **before** the annotated dates, the change
was at least partly voluntary rather than mandated — a crucial distinction
for policy conclusions.


In [ ]:
# ─── Gap 1: NPI event timeline ──────────────────────────────────────────────
# Uses default NPI dates for the selected region; override with npi_events={}.
# Requires the weekly_rg parquet table to be populated (Section 2 pipeline).

fig_gap1 = plt_obj.plot_gap1_npi_timeline(save=True)
plt.show()

# ── Standalone usage (custom events) ───────────────────────────────────────
# from src.gap_analysis_plots import plot_npi_timeline, NPI_EVENTS
# import datetime
# custom_events = {
#     "My custom order": datetime.date(2020, 3, 20),
# }
# week2party = plt_obj._build_week2rg_by_party()  # load data
# fig = plot_npi_timeline(week2party, npi_events=custom_events, region=REGION)


### 4.2 Gap 2 — Sampling bias quantification

Compares the number of users per county to the census population, revealing
which county sizes / income levels are over- or under-represented in the
opt-in Cuebiq sample.  An OLS slope < 1.0 in the left panel means large
counties are over-represented; income skew appears in the right panel.


In [ ]:
# ─── Gap 2: Sampling bias ───────────────────────────────────────────────────
# Uses the built-in rurality table (pop2023 + area) as a census proxy.
# Pass df_census= with an 'income_median' column for a richer analysis.

fig_gap2 = plt_obj.plot_gap2_sampling_bias(save=True)
plt.show()

# ── With income data (example) ──────────────────────────────────────────────
# import pandas as pd
# df_cen = pd.read_csv('census_data/income_california/economic_characteristics_california.csv')
# # ... parse and reshape to get county + income_median columns ...
# fig = plt_obj.plot_gap2_sampling_bias(df_census=df_cen, income_col='income_median')

# ── Quintile coverage plot ───────────────────────────────────────────────────
# from src.gap_analysis_plots import plot_sampling_bias_quintiles, compute_users_per_county
# dfs = plt_obj._load_scalars_all_periods()
# df_users = compute_users_per_county(dfs)
# fig_q = plot_sampling_bias_quintiles(df_users, dataset.df_rurality.rename(columns={'name':'county'}))


### 4.3 Gap 3 — Party / rurality conflation

In California, Republican-governed counties are disproportionately rural.
This plot runs:

1. **Box-plots** of the metric split by party × rurality.
2. **OLS regression** (`metric ~ party + rurality + party×rurality`) with
   95% CIs — requires `statsmodels`.
3. **Partial correlations** via residualisation — requires `scipy`.


In [ ]:
# ─── Gap 3: Party / rurality conflation ────────────────────────────────────
# Default metric: radius_gyration.  Swap for any SCALAR_METRICS_FLOAT name.

fig_gap3_rg = plt_obj.plot_gap3_party_rurality(metric='radius_gyration', save=True)
plt.show()

# ── Other metrics ────────────────────────────────────────────────────────────
# fig_gap3_ent = plt_obj.plot_gap3_party_rurality(metric='real_entropy')
# fig_gap3_dist = plt_obj.plot_gap3_party_rurality(metric='distance')


### 4.4 Gap 4 — Post-lockdown asymmetry (explicit Results figure)

The paper's *Conclusion* states that Democrats show reduced willingness to
resume mobility post-lockdown compared to Republicans, but no figure in
the *Results* section demonstrates this.  This cell produces that figure:

* **Left panel**: median metric per period by party (IQR shaded).
* **Right panel**: Δ metric vs. pre-lockdown baseline, grouped by party.


In [ ]:
# ─── Gap 4: Post-lockdown asymmetry ────────────────────────────────────────

fig_gap4_rg = plt_obj.plot_gap4_post_lockdown_asymmetry(metric='radius_gyration', save=True)
plt.show()

# ── Additional metrics from the paper ────────────────────────────────────────
# fig_gap4_ent  = plt_obj.plot_gap4_post_lockdown_asymmetry(metric='real_entropy')
# fig_gap4_dist = plt_obj.plot_gap4_post_lockdown_asymmetry(metric='distance')

# ── Standalone usage with a custom metric label ──────────────────────────────
# from src.gap_analysis_plots import plot_post_lockdown_asymmetry
# dfs = plt_obj._load_scalars_all_periods()
# fig = plot_post_lockdown_asymmetry(dfs, metric='radius_gyration',
#                                    metric_label='Radius of Gyration')


---
## 4 · MIGRATE EXISTING OUTPUT TO S3 (final_pipeline)

**When using MODE B (S3 progressive)**: the pipeline automatically uploads each
shard's output to S3 and deletes local files.  At the very end it also runs a
final consolidation step that merges all per-shard S3 files into a single
`consolidated.parquet` per metric kind per period.

**This section is only needed if:**
- The pipeline ended before the final consolidation ran (e.g. killed after the
  last shard but before the merge step), **or**
- You used MODE A or MODE C (local computation) and the automatic S3 upload in
  Section 2 was skipped or failed.

The canonical output root is:
```
chub-datalake / final_pipeline/{REGION}/
```

Run cells 4.1 and 4.2 as needed.  Already-uploaded files are not re-uploaded.


In [ ]:
# ─── 4.1  Finalise S3 consolidation (MODE B only) ───────────────────────────
# Call this if the pipeline exited after the last shard but before the final
# merge.  For each period it downloads all per-shard S3 files, merges them
# into consolidated.parquet, uploads to S3, and deletes the per-shard files.

UPLOAD_ENDPOINT = S3_ENDPOINT_URL
UPLOAD_BUCKET   = S3_OUTPUT_BUCKET
UPLOAD_PREFIX   = S3_OUTPUT_PREFIX[REGION]

print(f"S3 target : s3://{UPLOAD_BUCKET}/{UPLOAD_PREFIX}")
print(f"Endpoint  : {UPLOAD_ENDPOINT}")
print(f"Region    : {REGION}")

print("\nRunning final S3 consolidation for all periods …")
for period in PERIOD_NAMES:
    print(f"\n  Period: {period!r}")
    store.consolidate_s3_shards(
        period,
        s3_bucket           = UPLOAD_BUCKET,
        s3_prefix           = UPLOAD_PREFIX,
        endpoint_url        = UPLOAD_ENDPOINT,
        delete_shards_after = True,
        verbose             = True,
    )
print("\nFinal consolidation complete.")


In [ ]:
# ─── 4.2  Upload local output to S3 (MODE A / MODE C) ───────────────────────
# If you ran MODE A (local raw files) or MODE C (legacy migration), results are
# in the local ParquetStore.  Use this cell to push them to S3.

periods_on_s3 = store.list_s3_computed_periods(
    s3_bucket    = UPLOAD_BUCKET,
    s3_prefix    = UPLOAD_PREFIX,
    endpoint_url = UPLOAD_ENDPOINT,
)
periods_to_upload = [p for p in PERIOD_NAMES if p not in periods_on_s3]

print(f"Periods already on S3 : {periods_on_s3 or 'none'}")
print(f"Periods to upload     : {periods_to_upload or 'none'}")


In [ ]:
# ── Upload to S3 (MODE A / MODE C) ───────────────────────────────────────────
if not periods_to_upload:
    print("All periods already on S3. Nothing to upload.")
else:
    upload_results = store.upload_all_to_s3(
        period_names      = periods_to_upload,
        s3_bucket         = UPLOAD_BUCKET,
        s3_prefix         = UPLOAD_PREFIX,
        endpoint_url      = UPLOAD_ENDPOINT,
        delete_after      = True,
        consolidate_first = True,
        verbose           = True,
    )
    print("\nUpload summary:")
    for period, kinds in upload_results.items():
        for kind, ok in kinds.items():
            status = "✓" if ok else "✗"
            print(f"  {status}  {period:25s}  {kind}")


---
## 5 · TRANSITION MATRICES (geohash grid)

Compute presence and transition matrices from raw trajectories on a
geohash grid.  Results go directly to S3 — no persistent local storage.

**Pipeline overview:**
1. Tile the region with a geohash grid (`tile_counties_via_geohash`)
2. Load raw trajectories for each period
3. Coarsen trajectory geohashes to the chosen precision
4. Bin time into `delta_time_h`-hour intervals
5. Compute presence matrix (birth/death/transit counts per cell per bin)
6. Compute transition matrix (user moves between cells)
7. Write temp parquet → upload to S3 → delete temp

**Output location:**
```
chub-datalake / final_pipeline/{REGION}/transition_matrices/
    presence_prec5_dh1.0_<period>.parquet
    transition_prec5_dh1.0_<period>.parquet
    cache_index.json
```


In [ ]:
from src.tile_counties_via_geohash import tile_counties_via_geohash
from src.transition_matrices import TransitionPipeline
from src.constants import S3_TRANSITION_PREFIX

# ─── Geohash precision ───────────────────────────────────────────────────────
# Precision 5 → ~5km × 5km cells — recommended for CA/MA with 32 GB RAM.
# Precision 4 → ~39km × 20km cells — use for a lighter overview.
GEOHASH_PRECISION = 4
DELTA_TIME_H      = 1.0    # time-bin width in hours

# ─── Build geohash grid ──────────────────────────────────────────────────────
print(f"Building geohash grid (precision={GEOHASH_PRECISION}) for {REGION} …")
grid = tile_counties_via_geohash(dataset.geojson, precision=GEOHASH_PRECISION)
print(f"  Grid cells: {len(grid):,}")
print(grid.head(3))


In [ ]:
# ─── Initialise transition pipeline ─────────────────────────────────────────
tm_pipeline = TransitionPipeline(
    dataset           = dataset,
    geohash_precision = GEOHASH_PRECISION,
    delta_time_h      = DELTA_TIME_H,
    endpoint_url      = S3_ENDPOINT_URL,
    bucket            = S3_OUTPUT_BUCKET,
    s3_prefix         = S3_TRANSITION_PREFIX[REGION],
    temp_dir          = None,     # uses /tmp/humobcov_transitions by default
    keep_local        = False,    # delete temp files after successful upload
)

# ─── Show cache status (what is already on S3) ───────────────────────────────
tm_pipeline.summary()


In [ ]:
# ─── Run all periods ─────────────────────────────────────────────────────────
# Resume-safe: already-computed periods are skipped.
# Data flows: memory → temp parquet → S3 → delete local.
# Set force=True to recompute even if cached.

results = tm_pipeline.run_all_periods(force=False, verbose=True)

print("\nTransition matrix run summary:")
for period, kinds in results.items():
    for kind, ok in kinds.items():
        status = "✓" if ok else "✗ (temp file kept locally)"
        print(f"  {status}  {period:25s}  {kind}")
